In [1]:
#k1-b

!pip install -q duckdb networkx shapely tinydb "deltalake>=1.0"

from google.colab import drive
drive.mount("/content/drive")

DIR_P3 = "/content/drive/MyDrive/BigData/Praktikum3"
DIR_SIMPAN = "/content/drive/MyDrive/BigData/Praktikum4"
TRIPS_BERSIH = f"{DIR_P3}/trips_bersih"

# Hanya dipakai bila trips_bersih/ Anda hilang: membuat data tiruan dengan kolom yang sama.
import numpy as np, pandas as pd, os, shutil

def bangkitkan_trips_sintetis(keluar, n=1_000_000, seed=42):
    rng = np.random.default_rng(seed)
    wilayah = ["Manhattan", "Brooklyn", "Queens", "Bronx", "Staten Island", "EWR"]
    zona_ids = np.arange(1, 264)
    wilayah_zona = rng.choice(wilayah, size=len(zona_ids), p=[0.35, 0.25, 0.22, 0.10, 0.05, 0.03])
    zona = pd.DataFrame({"LocationID": zona_ids, "borough_naik": wilayah_zona})

    pop = np.clip(rng.zipf(1.6, size=len(zona_ids)).astype(float), 1, 60)
    pop = pop / pop.sum()

    pu = rng.choice(zona_ids, size=n, p=pop)
    do = rng.choice(zona_ids, size=n, p=pop)
    awal = pd.Timestamp("2023-01-01")
    pickup = awal + pd.to_timedelta(rng.integers(0, 31 * 24 * 60, n), unit="m")
    durasi = rng.integers(2, 90, n)
    dist = np.round(rng.gamma(2.0, 1.6, n).clip(0.05, 60), 2)
    fare = np.round(2.5 + dist * 2.8 + rng.gamma(2.0, 1.0, n), 2)
    tip = np.round(rng.gamma(1.5, 1.2, n), 2)
    pay = rng.choice([1, 2, 3, 4], size=n, p=[0.7, 0.27, 0.02, 0.01])

    d = pd.DataFrame({
        "tpep_pickup_datetime": pickup,
        "tpep_dropoff_datetime": pickup + pd.to_timedelta(durasi, unit="m"),
        "passenger_count": rng.integers(1, 5, n).astype("float64"),
        "trip_distance": dist, "PULocationID": pu, "DOLocationID": do,
        "payment_type": pay, "fare_amount": fare, "tip_amount": tip,
        "total_amount": np.round(fare + tip + 3.0, 2),
        "durasi_menit": durasi.astype("float64"),
        "nama_pembayaran": pd.Series(pay).map({1: "Kartu kredit", 2: "Tunai", 3: "Gratis", 4: "Sengketa"}).values,
    })

    d = d.merge(zona.rename(columns={"LocationID": "PULocationID"}), on="PULocationID", how="left")
    d["suhu_c"] = np.round(rng.normal(3, 5, n), 1)
    d["hujan_mm"] = np.round(rng.gamma(0.4, 0.8, n), 2)

    if os.path.exists(keluar):
        shutil.rmtree(keluar)
    d.to_parquet(keluar, partition_cols=["borough_naik"], index=False)
    return keluar

# Buka komentar baris di bawah untuk memakainya, lalu lanjut ke K-1 (sel kode umum):
TRIPS_BERSIH = bangkitkan_trips_sintetis("/content/trips_bersih_sintetis")

import os, time, json, sqlite3, random, shutil, glob
import numpy as np, pandas as pd

# Pengukur waktu (dari Praktikum 1/K-3, versi ringkas)
catatan = []
def ukur(label, fungsi):
    t0 = time.perf_counter()
    hasil = fungsi()
    detik = time.perf_counter() - t0
    catatan.append({"langkah": label, "detik": round(detik, 4)})
    print(f"[{label}] {detik:.4f} s")
    return hasil

df = ukur("baca trips_bersih", lambda: pd.read_parquet(TRIPS_BERSIH))

# Ganti nama kolom asli data TLC menjadi nama yang lebih mudah dibaca
NAMA_BARU = {
    "tpep_pickup_datetime": "waktu_jemput",
    "PULocationID": "zona_jemput",
    "DOLocationID": "zona_tujuan",
    "borough_naik": "wilayah_jemput",
    "trip_distance": "jarak_mil",
    "total_amount": "total_bayar",
    "tip_amount": "tip",
    "nama_pembayaran": "metode_bayar",
}
df = df.rename(columns=NAMA_BARU)
df["wilayah_jemput"] = df["wilayah_jemput"].astype(str)
print("baris:", f"{len(df):,}", "| kolom:", df.shape[1])

# Contoh 300.000 baris untuk K-2 dan K-3 (agar tidak terlalu lama)
sample = df.sample(n=min(300_000, len(df)), random_state=42).reset_index(drop=True)
sample["id_perjalanan"] = sample.index + 1
sample["waktu_jemput"] = sample["waktu_jemput"].astype(str)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 MB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 34.6 MB/s eta 0:00:00
Mounted at /content/drive
[baca trips_bersih] 0.4065 s
baris: 1,000,000 | kolom: 15


Bagian ini merupakan langkah operasional cadangan yang hanya dijalankan apabila data hasil praktikum sebelumnya tidak dapat ditemukan. Kode akan membangkitkan data sintetis yang sengaja dirancang untuk meniru struktur dan distribusi nilai dari data taksi aslinya. Data tiruan tersebut kemudian disimpan ulang dalam format yang sesuai agar seluruh rangkaian praktikum dapat tetap dilanjutkan tanpa hambatan.

In [2]:
#k2

KOLOM = ["id_perjalanan", "waktu_jemput", "zona_jemput", "zona_tujuan",
         "wilayah_jemput", "jarak_mil", "total_bayar", "metode_bayar"]
data = sample[KOLOM].copy()

# Model 1 relasional: tabel SQLite
if os.path.exists("uji.db"):
    os.remove("uji.db")
con = sqlite3.connect("uji.db")
ukur("relasional: muat tabel", lambda: data.to_sql("perjalanan", con, index=False))

# Model 2 key-value: kamus (dict) dengan kunci "perjalanan:<id>", nilainya teks JSON
def bangun_kv():
    return {f"perjalanan:{r['id_perjalanan']}": json.dumps(r) for r in data.to_dict("records")}

kv = ukur("key-value: bangun dict", bangun_kv)
print("jumlah key:", len(kv))

# Pertanyaan A: ambil SATU perjalanan berdasarkan id (200 id acak)
random.seed(42)
id_uji = random.sample(range(1, len(data) + 1), 200)

def cari_sql():
    cur = con.cursor()
    return [cur.execute("SELECT * FROM perjalanan WHERE id_perjalanan = ?", (i,)).fetchone() for i in id_uji]

def cari_kv():
    return [json.loads(kv[f"perjalanan:{i}"]) for i in id_uji]

r1 = ukur("cari 200 perjalanan SQL tanpa indeks", cari_sql)
con.execute("CREATE INDEX idx_id ON perjalanan(id_perjalanan)") # buat indeks
r2 = ukur("cari 200 perjalanan SQL dengan indeks", cari_sql)
r3 = ukur("cari 200 perjalanan key-value", cari_kv)
print("hasil sama?", r1[0][0] == r2[0][0] == r3[0]["id_perjalanan"])

# Pertanyaan B: hitung rata-rata total bayar per wilayah (melibatkan SEMUA data)
def hitung_sql():
    return con.execute("""SELECT wilayah_jemput, COUNT(*), ROUND(AVG(total_bayar), 2)
                          FROM perjalanan GROUP BY wilayah_jemput ORDER BY 2 DESC, 1""").fetchall()

def hitung_kv():
    total, jumlah = {}, {}
    for teks in kv.values():
        d = json.loads(teks) # setiap nilai harus dibuka dulu
        w = d["wilayah_jemput"]
        total[w] = total.get(w, 0) + d["total_bayar"]
        jumlah[w] = jumlah.get(w, 0) + 1
    return sorted([(w, jumlah[w], round(total[w] / jumlah[w], 2)) for w in jumlah],
                  key=lambda x: (-x[1], x[0]))

h1 = ukur("hitung per wilayah SQL GROUP BY", hitung_sql)
h2 = ukur("hitung per wilayah key-value (buka semua nilai)", hitung_kv)
print(h1[:3])
print("hasil sama?", h1 == h2)

[relasional: muat tabel] 1.8522 s
[key-value: bangun dict] 9.6918 s
jumlah key: 300000
[cari 200 perjalanan SQL tanpa indeks] 8.8441 s
[cari 200 perjalanan SQL dengan indeks] 0.0062 s
[cari 200 perjalanan key-value] 0.0020 s
hasil sama? True
[hitung per wilayah SQL GROUP BY] 0.2942 s
[hitung per wilayah key-value (buka semua nilai)] 1.9847 s
[('Manhattan', 116930, 18.26), ('Queens', 81377, 18.29), ('Brooklyn', 62038, 18.24)]
hasil sama? True


Langkah ini membandingkan kinerja penyimpanan data menggunakan struktur tabel relasional konvensional dan struktur penyusunan kunci nilai. Percobaan pertama mengukur efisiensi kecepatan pencarian satu data spesifik menggunakan indeks dibandingkan dengan pencarian biasa yang tanpa indeks. Percobaan kedua menyoroti kelemahan struktur kunci nilai saat digunakan untuk menghitung nilai rata rata karena setiap baris dokumen harus dibuka secara manual satu per satu.

In [3]:
#k3

from tinydb import TinyDB, Query
from tinydb.storages import MemoryStorage

def ke_dokumen(r):
    dok = {
        "id_perjalanan": int(r.id_perjalanan),
        "waktu_jemput": r.waktu_jemput,
        "jemput": {"zona": int(r.zona_jemput), "wilayah": r.wilayah_jemput},
        "tujuan": {"zona": int(r.zona_tujuan)},
        "biaya": {"jarak_mil": float(r.jarak_mil),
                  "total": float(r.total_bayar),
                  "metode_bayar": r.metode_bayar},
    }
    # Tip hanya dicatat untuk pembayaran kartu -> field ini ada di sebagian dokumen saja
    if r.metode_bayar == "Kartu kredit":
        dok["biaya"]["tip"] = float(r.tip)
    return dok

db = TinyDB(storage=MemoryStorage)
tabel = db.table("perjalanan")
docs = [ke_dokumen(r) for r in sample.head(20_000).itertuples()]
tabel.insert_multiple(docs)

print("jumlah dokumen:", len(tabel))
print(json.dumps(tabel.all()[0], indent=2))

Q = Query()
print("jemput di Manhattan & total > 50:", len(tabel.search((Q.jemput.wilayah == "Manhattan") & (Q.biaya.total > 50))))
print("dokumen yang punya field tip :", len(tabel.search(Q.biaya.tip.exists())))
print("dokumen yang TIDAK punya tip :", len(tabel.search(~Q.biaya.tip.exists())))

# Menambah field baru ("cuaca") pada satu dokumen tanpa mengubah struktur apa pun
tabel.insert({"id_perjalanan": 999_999, "waktu_jemput": "2023-01-15 08:00:00",
              "jemput": {"zona": 161, "wilayah": "Manhattan"}, "tujuan": {"zona": 236},
              "biaya": {"jarak_mil": 1.2, "total": 14.5, "metode_bayar": "Kartu kredit", "tip": 3.0},
              "cuaca": {"hujan": True, "suhu_c": -2.0}})
print("dokumen yang punya field cuaca :", len(tabel.search(Q.cuaca.exists())))
print("total dokumen sekarang :", len(tabel))

jumlah dokumen: 20000
{
  "id_perjalanan": 1,
  "waktu_jemput": "2023-01-10 17:57:00",
  "jemput": {
    "zona": 219,
    "wilayah": "Staten Island"
  },
  "tujuan": {
    "zona": 46
  },
  "biaya": {
    "jarak_mil": 2.0,
    "total": 17.99,
    "metode_bayar": "Kartu kredit",
    "tip": 4.81
  }
}
jemput di Manhattan & total > 50: 9
dokumen yang punya field tip : 13979
dokumen yang TIDAK punya tip : 6021
dokumen yang punya field cuaca : 1
total dokumen sekarang : 20001


Kode ini menunjukkan penggunaan pustaka basis data ringan untuk menyimpan informasi dalam bentuk dokumen berformat objek JavaScript yang bersarang. Keuntungan utama dari pendekatan ini adalah kemampuan sistem untuk menambahkan atribut baru pada suatu dokumen tertentu tanpa harus mengubah skema tabel secara keseluruhan. Meskipun pendekatannya sangat fleksibel, metode ini menuntut perancang aplikasi untuk melakukan validasi datanya sendiri karena basis data tidak menegakkan aturan skema secara otomatis.

In [4]:
#k4

import duckdb

GLOB = f"{TRIPS_BERSIH}/*/*.parquet"
con_d = duckdb.connect()

# VIEW = "jendela" bernama yang membaca Parquet langsung (tidak menyalin data).
# Di sinilah nama kolom asli TLC diberi nama yang mudah dibaca.
con_d.sql(f"""
CREATE VIEW perjalanan AS
SELECT tpep_pickup_datetime AS waktu_jemput,
       PULocationID AS zona_jemput,
       DOLocationID AS zona_tujuan,
       borough_naik AS wilayah_jemput,
       trip_distance AS jarak_mil,
       total_amount AS total_bayar,
       nama_pembayaran AS metode_bayar
FROM read_parquet('{GLOB}', hive_partitioning=true)""")

SQL = """SELECT wilayah_jemput, COUNT(*) AS jumlah, ROUND(AVG(total_bayar), 2) AS rata_bayar
         FROM perjalanan GROUP BY wilayah_jemput ORDER BY jumlah DESC, wilayah_jemput"""
hasil_d = ukur("DuckDB hitung per wilayah (langsung dari Parquet)", lambda: con_d.sql(SQL).df())
print(hasil_d)

# Pembanding: SQLite (menyimpan per baris) pada data yang sama
KOLOM_S = ["wilayah_jemput", "total_bayar", "jarak_mil", "zona_jemput", "zona_tujuan"]
if os.path.exists("baris.db"):
    os.remove("baris.db")
con_s = sqlite3.connect("baris.db")

ukur("SQLite muat seluruh baris ke tabel", lambda: df[KOLOM_S].to_sql("perjalanan", con_s, index=False))
hasil_s = ukur("SQLite hitung per wilayah", lambda: con_s.execute(
    """SELECT wilayah_jemput, COUNT(*), ROUND(AVG(total_bayar), 2)
       FROM perjalanan GROUP BY wilayah_jemput ORDER BY 2 DESC, 1""").fetchall())
print("hasil sama?", [tuple(r) for r in hasil_d.itertuples(index=False)] == [tuple(r) for r in hasil_s])

# Filter pada kolom partisi: DuckDB hanya membuka folder wilayah yang diminta (partition pruning)
SQL2 = """SELECT COUNT(*), ROUND(AVG(total_bayar), 2)
          FROM perjalanan WHERE wilayah_jemput = 'Staten Island'"""
print(ukur("DuckDB hanya wilayah Staten Island", lambda: con_d.sql(SQL2).fetchall()))

ukuran_pq = sum(os.path.getsize(os.path.join(r, f)) for r, d, fs in os.walk(TRIPS_BERSIH) for f in fs) / 1024**2
print(f"Parquet ({df.shape[1]} kolom): {ukuran_pq:.1f} MB | SQLite ({len(KOLOM_S)} kolom): {os.path.getsize('baris.db') / 1024**2:.1f} MB")

[DuckDB hitung per wilayah (langsung dari Parquet)] 0.0617 s
  wilayah_jemput  jumlah  rata_bayar
0      Manhattan  391107       18.26
1         Queens  270929       18.27
2       Brooklyn  205487       18.29
3  Staten Island   67041       18.28
4          Bronx   60912       18.21
5            EWR    4524       18.31
[SQLite muat seluruh baris ke tabel] 6.0976 s
[SQLite hitung per wilayah] 1.7459 s
hasil sama? True
[DuckDB hanya wilayah Staten Island] 0.0395 s
[(67041, 18.28)]
Parquet (15 kolom): 32.2 MB | SQLite (5 kolom): 37.6 MB


Bagian ini mendemonstrasikan kemampuan mesin analitik dalam menjalankan kueri secara langsung di atas kumpulan berkas tanpa perlu memuat data ke dalam tabel. Penyimpanan berbasis kolom ini terbukti memiliki waktu proses yang jauh lebih cepat untuk operasi agregasi dibandingkan dengan penyimpanan berbasis baris pada umumnya. Kinerja yang cepat ini dapat terjadi karena mesin pembaca hanya mengakses kolom yang sedang dibutuhkan dan memanfaatkan informasi partisi untuk melewati data yang tidak relevan.

In [5]:
#k5

import networkx as nx

# Satu baris = satu arus: berapa perjalanan dari zona jemput ke zona tujuan tertentu
arus = ukur("hitung arus antar zona", lambda: con_d.sql("""
    SELECT zona_jemput AS asal, zona_tujuan AS tujuan, COUNT(*) AS jumlah
    FROM perjalanan GROUP BY 1, 2""").df())

# Zona = node (titik), arus = edge (garis berarah, bobotnya jumlah perjalanan)
G = nx.DiGraph()
for r in arus.itertuples(index=False):
    G.add_edge(int(r.asal), int(r.tujuan), jumlah=int(r.jumlah))
print("node (zona):", G.number_of_nodes(), " edge (arus):", G.number_of_edges())

# Lima arus terbesar. Urutan dibuat pasti dengan pengurut tambahan (jumlah, asal, tujuan)
top_graf = sorted(G.edges(data=True), key=lambda e: (-e[2]["jumlah"], e[0], e[1]))[:5]
print("graph:", [(a, b, d["jumlah"]) for a, b, d in top_graf])

top_sql = arus.sort_values(["jumlah", "asal", "tujuan"], ascending=[False, True, True]).head(5)
print("SQL", list(top_sql.itertuples(index=False, name=None)))

# Lima zona yang paling banyak menjadi tujuan
populer = sorted(G.in_degree(weight="jumlah"), key=lambda kv: (-kv[1], kv[0]))[:5]
print("zona tujuan terpopuler:", populer)

MIN = 50
zona_awal = populer[0][0] # mulai dari zona tujuan terpopuler

# Pertanyaan: zona mana yang bisa dicapai dalam TEPAT 2 langkah dari zona_awal?
# (1) cara graph: telusuri tetangga dari tetangga
def dua_langkah_graph():
    hasil = set()
    for b, d1 in G[zona_awal].items():
        if d1["jumlah"] < MIN:
            continue
        for c, d2 in G[b].items():
            if d2["jumlah"] >= MIN and c != zona_awal:
                hasil.add(c)
    return hasil

# (2) cara SQL: harus menggabungkan tabel arus dengan dirinya sendiri (self-join)
con_d.register("arus", arus)
def dua_langkah_sql():
    return set(con_d.sql(f"""
        SELECT DISTINCT a2.tujuan
        FROM arus a1 JOIN arus a2 ON a1.tujuan = a2.asal
        WHERE a1.asal = {zona_awal} AND a1.jumlah >= {MIN}
        AND a2.jumlah >= {MIN} AND a2.tujuan <> {zona_awal}""").df()["tujuan"])

hg = ukur("2 langkah graph (NetworkX)", dua_langkah_graph)
hs = ukur("2 langkah SQL self-join (DuckDB)", dua_langkah_sql)
print("zona awal:", zona_awal, " terjangkau (graph):", len(hg), "| (SQL):", len(hs), " sama?", hg == hs)

# (3) Jumlah langkah bebas: "dalam paling banyak k langkah" cukup ganti angka k
H = nx.DiGraph([(a, b) for a, b, d in G.edges(data=True) if d["jumlah"] >= MIN])
for k in (1, 2, 3, 4):
    n = len(nx.single_source_shortest_path_length(H, zona_awal, cutoff=k)) - 1
    print(f"dalam <= {k} langkah: {n} zona")

[hitung arus antar zona] 0.1402 s
node (zona): 263  edge (arus): 44120
graph: [(46, 205, 789), (146, 129, 787), (74, 221, 765), (221, 166, 765), (46, 221, 761)]
SQL [(46, 205, 789), (146, 129, 787), (74, 221, 765), (221, 166, 765), (46, 221, 761)]
zona tujuan terpopuler: [(183, 26941), (104, 26916), (146, 26891), (164, 26867), (194, 26847)]
[2 langkah graph (NetworkX)] 0.0204 s
[2 langkah SQL self-join (DuckDB)] 0.0197 s
zona awal: 183  terjangkau (graph): 91 | (SQL): 91  sama? True
dalam <= 1 langkah: 83 zona
dalam <= 2 langkah: 91 zona
dalam <= 3 langkah: 91 zona
dalam <= 4 langkah: 91 zona


Kode pada langkah ini memodelkan pergerakan taksi antar wilayah sebagai sebuah struktur graf berarah. Titik pada graf merepresentasikan zona lokasi penjemputan atau tujuan, sedangkan garis penghubungnya menunjukkan besaran volume perjalanan di antara kedua titik tersebut. Pemodelan berbasis graf ini sangat efisien untuk menjawab berbagai pertanyaan terkait keterjangkauan rute wilayah jika dibandingkan dengan metode kueri penggabungan tabel yang cukup membebani kinerja.

In [6]:
# k6

from shapely import STRtree, box, points

# CATATAN: data TLC 2023 hanya berisi ID zona, TANPA koordinat. Titik di bawah ini SINTETIS.
rng = np.random.default_rng(42)
N = 200_000
bujur = rng.normal(-73.97, 0.06, N).clip(-74.25, -73.70)   # longitude
lintang = rng.normal(40.74, 0.05, N).clip(40.50, 40.92)    # latitude
titik = points(bujur, lintang)

# 1.000 wilayah pencarian berupa kotak kecil (sekitar 1 km x 1 km)
M = 1000
cx = rng.uniform(-74.05, -73.85, M)
cy = rng.uniform(40.65, 40.85, M)
kotak = [box(x - 0.005, y - 0.005, x + 0.005, y + 0.005) for x, y in zip(cx, cy)]

# Cara 1: periksa titik satu per satu dengan loop Python (hanya 5 kotak, 1.000 akan sangat lama)
def cara_loop():
    return [sum(1 for p in titik if k.contains(p)) for k in kotak[:5]]

# Cara 2: bandingkan seluruh titik sekaligus dengan NumPy (tanpa indeks spasial), 1.000 kotak
def cara_numpy():
    hasil = []
    for x, y in zip(cx, cy):
        m = (bujur >= x - 0.005) & (bujur <= x + 0.005) & (lintang >= y - 0.005) & (lintang <= y + 0.005)
        hasil.append(int(m.sum()))
    return hasil

h_loop = ukur("cara 1 loop Python, 5 kotak", cara_loop)
h_np = ukur("cara 2 NumPy, 1000 kotak", cara_numpy)

# Cara 3: indeks spasial STRtree (sejenis R-tree)
pohon = ukur("bangun indeks STRtree", lambda: STRtree(titik))
h_idx = ukur("cara 3 STRtree, 1000 kotak", lambda: [len(pohon.query(k, predicate="contains")) for k in kotak])

print("5 kotak pertama loop:", h_loop, "| STRtree:", h_idx[:5])
print("hasil NumPy dan STRtree sama?", h_np == h_idx, "| loop dan STRtree sama (5 pertama)?", h_loop == h_idx[:5])
print("total titik ditemukan (1000 kotak):", sum(h_idx))

[cara 1 loop Python, 5 kotak] 13.1894 s
[cara 2 NumPy, 1000 kotak] 0.5543 s
[bangun indeks STRtree] 0.0751 s
[cara 3 STRtree, 1000 kotak] 0.3765 s
5 kotak pertama loop: [1012, 84, 228, 748, 509] | STRtree: [1012, 84, 228, 748, 509]
hasil NumPy dan STRtree sama? True | loop dan STRtree sama (5 pertama)? True
total titik ditemukan (1000 kotak): 417545


Langkah ini berfokus pada teknik pencarian sebaran titik koordinat di dalam batasan suatu wilayah dengan memanfaatkan bantuan indeks spasial. Penggunaan indeks spasial ini sangat memungkinkan proses pencarian lokasi berjalan secara instan karena sistem memotong wilayah dan tidak perlu memeriksa setiap titik koordinat secara satu per satu. Hasil waktu eksekusinya kemudian dibandingkan secara langsung dengan metode pengulangan pencarian biasa untuk membuktikan letak tingkat efisiensi penggunaan indeks pada data berbasis lokasi.

In [7]:
# k7

from deltalake import DeltaTable, write_deltalake
import os
import shutil

# Opsi tambahan khusus untuk Google Drive di Colab
opsi_drive = {"allow_unsafe_rename": "true"} # <--- TAMBAHKAN INI

KOLOM_DELTA = ["waktu_jemput", "zona_jemput", "zona_tujuan",
               "wilayah_jemput", "jarak_mil", "total_bayar", "metode_bayar"]
data_delta = df[KOLOM_DELTA].sample(n=min(200_000, len(df)), random_state=42).reset_index(drop=True)
PATH_DELTA = f"{DIR_SIMPAN}/trips_delta"

if os.path.exists(PATH_DELTA):
    shutil.rmtree(PATH_DELTA)

# Versi 0: menulis data awal (100.000 baris)
# Tambahkan storage_options di bawah ini
write_deltalake(PATH_DELTA, data_delta.iloc[:100_000], mode="overwrite", storage_options=opsi_drive) # <--- DIUBAH
dt = DeltaTable(PATH_DELTA, storage_options=opsi_drive) # <--- DIUBAH
print("versi:", dt.version(), "| baris:", len(dt.to_pandas()))

# Versi 1: menambah 50.000 baris
write_deltalake(PATH_DELTA, data_delta.iloc[100_000:150_000], mode="append", storage_options=opsi_drive) # <--- DIUBAH
dt = DeltaTable(PATH_DELTA, storage_options=opsi_drive) # <--- DIUBAH
print("versi:", dt.version(), "| baris:", len(dt.to_pandas()))

# Versi 2: koreksi data
sekarang = data_delta.iloc[:150_000]
n_koreksi = int(((sekarang["wilayah_jemput"] == "Bronx") & (sekarang["metode_bayar"] == "Tunai")).sum())
print("baris yang akan dikoreksi:", n_koreksi)

dt.update(predicate="wilayah_jemput = 'Bronx' AND metode_bayar = 'Tunai'",
          updates={"total_bayar": "total_bayar + 1.0"})
print("versi:", DeltaTable(PATH_DELTA, storage_options=opsi_drive).version()) # <--- DIUBAH

# TIME TRAVEL: baca tabel sebagaimana keadaannya pada versi tertentu
v0 = DeltaTable(PATH_DELTA, version=0, storage_options=opsi_drive).to_pandas() # <--- DIUBAH
v1 = DeltaTable(PATH_DELTA, version=1, storage_options=opsi_drive).to_pandas() # <--- DIUBAH
terbaru = DeltaTable(PATH_DELTA, storage_options=opsi_drive).to_pandas() # <--- DIUBAH

print("baris v0:", len(v0), "| v1:", len(v1), " terbaru:", len(terbaru))
selisih = terbaru["total_bayar"].sum() - v1["total_bayar"].sum()
print("selisih total_bayar (terbaru - v1):", round(selisih, 2), " jumlah baris dikoreksi:", n_koreksi)

# Riwayat perubahan tabel
for h in DeltaTable(PATH_DELTA, storage_options=opsi_drive).history(): # <--- DIUBAH
    print(h.get("version"), h.get("operation"))

# Log transaksi: satu file JSON per versi
print(sorted(os.listdir(f"{PATH_DELTA}/_delta_log")))

# PENEGAKAN SKEMA: menambah data dengan kolom asing harus DITOLAK
tambahan = data_delta.iloc[150_000:150_010].copy()
tambahan["kolom_asing"] = 1
try:
    write_deltalake(PATH_DELTA, tambahan, mode="append", storage_options=opsi_drive) # <--- DIUBAH
    print("append kolom asing: BERHASIL (tidak diharapkan)")
except Exception as e:
    print("append kolom asing DITOLAK:", type(e).__name__)
print("versi setelah percobaan gagal:", DeltaTable(PATH_DELTA, storage_options=opsi_drive).version()) # <--- DIUBAH

versi: 0 | baris: 100000
versi: 1 | baris: 150000
baris yang akan dikoreksi: 2517
versi: 2
baris v0: 100000 | v1: 150000  terbaru: 150000
selisih total_bayar (terbaru - v1): 2517.0  jumlah baris dikoreksi: 2517
2 UPDATE
1 WRITE
0 WRITE
['00000000000000000000.json', '00000000000000000001.json', '00000000000000000002.json']
append kolom asing DITOLAK: SchemaMismatchError
versi setelah percobaan gagal: 2


Bagian ini menerapkan format arsitektur penyimpanan modern untuk menambahkan fitur keandalan transaksi dan penegakan validasi skema pada penyimpanan data. Kode menunjukkan cara memperbarui data secara aman serta menguji kemampuan sistem dalam membaca kondisi data versi sebelumnya melalui fitur penjelajahan riwayat waktu. Selain itu sistem tersebut juga diperlihatkan mampu secara otomatis menolak masukan data baru yang membawa kolom asing demi menjaga konsistensi struktur tabel secara keseluruhan.

In [8]:
# k8

# Pastikan opsi_drive sudah terdefinisi
opsi_drive = {"allow_unsafe_rename": "true"}

dt = DeltaTable(PATH_DELTA, storage_options=opsi_drive)
print("file data aktif sebelum:", len(dt.file_uris()), "| versi:", dt.version())

# 20 kali menambah data kecil (500 baris)
for i in range(20):
    potongan = data_delta.iloc[150_000 + i * 500 : 150_000 + (i + 1) * 500]
    write_deltalake(PATH_DELTA, potongan, mode="append", storage_options=opsi_drive)

dt = DeltaTable(PATH_DELTA, storage_options=opsi_drive)
print("file data aktif setelah 20 append kecil:", len(dt.file_uris()), "| versi:", dt.version(), "| baris:", len(dt.to_pandas()))

t0 = time.perf_counter()
DeltaTable(PATH_DELTA, storage_options=opsi_drive).to_pandas()
t_sebelum = time.perf_counter() - t0

# COMPACTION: Cukup panggil tanpa parameter tambahan, otomatis mewarisi opsi dari objek 'dt'
hasil = dt.optimize.compact() # <--- DIUBAH (Dibersihkan dari custom_metadata)
print("hasil compact:", {k: hasil[k] for k in ("numFilesAdded", "numFilesRemoved")})
dt = DeltaTable(PATH_DELTA, storage_options=opsi_drive)

t0 = time.perf_counter()
_ = DeltaTable(PATH_DELTA, storage_options=opsi_drive).to_pandas()
t_sesudah = time.perf_counter() - t0

print("file data aktif setelah compact:", len(dt.file_uris()), " versi:", dt.version(), "| baris:", len(dt.to_pandas()))
print(f"waktu baca sebelum compact: {t_sebelum:.4f} s | sesudah: {t_sesudah:.4f} s")

# File lama TIDAK langsung terhapus (masih dibutuhkan untuk time travel). VACUUM yang menghapusnya.
print("file parquet di disk:", len(glob.glob(f"{PATH_DELTA}/*.parquet")))

# Jalankan vacuum dry run dengan aman
kandidat = dt.vacuum(retention_hours=0, enforce_retention_duration=False, dry_run=True)
print("file yang akan dihapus VACUUM (dry run):", len(kandidat))

file data aktif sebelum: 1 | versi: 2
file data aktif setelah 20 append kecil: 21 | versi: 22 | baris: 160000
hasil compact: {'numFilesAdded': 1, 'numFilesRemoved': 21}
file data aktif setelah compact: 1  versi: 23 | baris: 160000
waktu baca sebelum compact: 0.1587 s | sesudah: 0.1276 s
file parquet di disk: 24
file yang akan dihapus VACUUM (dry run): 23


Kode ini secara khusus mensimulasikan masalah penurunan performa yang sering muncul akibat penambahan data berukuran kecil secara berulang ulang hingga menghasilkan banyak tumpukan berkas. Permasalahan tersebut kemudian diselesaikan melalui sebuah operasi pemadatan yang bertugas menggabungkan banyak berkas kecil itu menjadi sedikit berkas namun berukuran besar. Berkas berkas berukuran kecil yang lama kemudian dibiarkan tetap ada di dalam penyimpanan untuk sementara waktu dan hanya akan dihapus secara permanen menggunakan perintah pembersihan khusus.

In [9]:
#k9

rekap = pd.DataFrame(catatan)
rekap.to_csv(f"{DIR_SIMPAN}/benchmark_nosql.csv", index=False)
print(rekap.to_string(index=False))

                                          langkah   detik
                                baca trips_bersih  0.4065
                           relasional: muat tabel  1.8522
                           key-value: bangun dict  9.6918
             cari 200 perjalanan SQL tanpa indeks  8.8441
            cari 200 perjalanan SQL dengan indeks  0.0062
                    cari 200 perjalanan key-value  0.0020
                  hitung per wilayah SQL GROUP BY  0.2942
  hitung per wilayah key-value (buka semua nilai)  1.9847
DuckDB hitung per wilayah (langsung dari Parquet)  0.0617
               SQLite muat seluruh baris ke tabel  6.0976
                        SQLite hitung per wilayah  1.7459
               DuckDB hanya wilayah Staten Island  0.0395
                           hitung arus antar zona  0.1402
                       2 langkah graph (NetworkX)  0.0204
                 2 langkah SQL self-join (DuckDB)  0.0197
                      cara 1 loop Python, 5 kotak 13.1894
              

Langkah yang terakhir ini berfungsi untuk mengumpulkan seluruh catatan perekaman waktu eksekusi dari percobaan sebelumnya ke dalam sebuah kerangka penyajian data. Data rekapitulasi tersebut kemudian diekspor ke dalam bentuk format nilai yang dipisahkan koma sebagai bahan lampiran laporan akhir praktikum mahasiswa. Proses pembuatan rangkuman ini diharap mempermudah mahasiswa dalam menganalisis dan menentukan perancangan model penyimpanan mana yang dinilai paling optimal untuk diterapkan pada berbagai jenis kebutuhan sistem nyata.

In [11]:
# latihan section

# Latihan 1: Pencarian 2000 ID acak
print("--- Latihan 1 ---")
random.seed(100)
id_uji_2000 = random.sample(range(1, len(data) + 1), 2000)

def cari_sql_2000():
    cur = con.cursor()
    return [cur.execute("SELECT * FROM perjalanan WHERE id_perjalanan = ?", (i,)).fetchone() for i in id_uji_2000]

def cari_kv_2000():
    return [json.loads(kv[f"perjalanan:{i}"]) for i in id_uji_2000]

waktu_sql = ukur("Latihan 1: SQL dengan indeks (2000 ID)", cari_sql_2000)
waktu_kv = ukur("Latihan 1: Key-value (2000 ID)", cari_kv_2000)

# Menghitung rata-rata mikrodetik (1 detik = 1.000.000 mikrodetik)
mikro_sql = (catatan[-2]["detik"] / 2000) * 1_000_000
mikro_kv = (catatan[-1]["detik"] / 2000) * 1_000_000
print(f"Rata-rata per pencarian -> SQL Indeks: {mikro_sql:.2f} mikrodetik | Key-Value: {mikro_kv:.2f} mikrodetik\n")

# Latihan 2: Hitung dokumen zona tertentu & Tunai
print("--- Latihan 2 ---")
# Menggunakan zona tujuan 46 sebagai contoh
ZONA_CEK = 46
hasil_dokumen = len(tabel.search((Q.tujuan.zona == ZONA_CEK) & (Q.biaya.metode_bayar == "Tunai")))
hasil_sql = con.execute("SELECT COUNT(*) FROM perjalanan WHERE zona_tujuan = ? AND metode_bayar = 'Tunai'", (ZONA_CEK,)).fetchone()[0]

print(f"Jumlah di TinyDB (Document): {hasil_dokumen}")
print(f"Jumlah di SQLite (Tabel): {hasil_sql}")
print(f"Apakah sama? {hasil_dokumen == hasil_sql}\n")

# Latihan 3: SUM satu kolom vs SUM beberapa kolom (DuckDB)
print("--- Latihan 3 ---")
q_satu = "SELECT SUM(total_bayar) FROM perjalanan"
q_banyak = "SELECT SUM(jarak_mil), SUM(total_bayar) FROM perjalanan"

ukur("Latihan 3: SUM 1 kolom (total_bayar)", lambda: con_d.sql(q_satu).df())
ukur("Latihan 3: SUM 2 kolom (jarak, total)", lambda: con_d.sql(q_banyak).df())
print()

# Latihan 4: 5 Zona dengan tujuan berbeda terbanyak (Graph vs SQL)
print("--- Latihan 4 ---")
# Graph: Menggunakan out_degree tanpa bobot (menghitung jumlah cabang unik/edge keluar)
top_out_graph = sorted(G.out_degree(), key=lambda x: (-x[1], x[0]))[:5]
print("Graph (out-degree):", top_out_graph)

# SQL: Menggunakan COUNT(DISTINCT)
top_out_sql = con_d.sql("""
    SELECT asal, COUNT(DISTINCT tujuan) AS jumlah_unik
    FROM arus GROUP BY asal ORDER BY jumlah_unik DESC, asal LIMIT 5
""").fetchall()
print("SQL (COUNT DISTINCT):", top_out_sql)
print(f"Apakah urutannya sama? {[x[0] for x in top_out_graph] == [x[0] for x in top_out_sql]}\n")

# Latihan 5: STRtree vs NumPy pada ukuran kotak berbeda
print("--- Latihan 5 ---")
# Kotak 0.02 derajat (Besar)
kotak_besar = [box(x - 0.01, y - 0.01, x + 0.01, y + 0.01) for x, y in zip(cx, cy)]
# Kotak 0.001 derajat (Kecil)
kotak_kecil = [box(x - 0.0005, y - 0.0005, x + 0.0005, y + 0.0005) for x, y in zip(cx, cy)]

def np_besar(): return [int(((bujur >= x - 0.01) & (bujur <= x + 0.01) & (lintang >= y - 0.01) & (lintang <= y + 0.01)).sum()) for x, y in zip(cx, cy)]
def np_kecil(): return [int(((bujur >= x - 0.0005) & (bujur <= x + 0.0005) & (lintang >= y - 0.0005) & (lintang <= y + 0.0005)).sum()) for x, y in zip(cx, cy)]

ukur("Latihan 5: NumPy Kotak 0.02", np_besar)
ukur("Latihan 5: STRtree Kotak 0.02", lambda: [len(pohon.query(k, predicate="contains")) for k in kotak_besar])

ukur("Latihan 5: NumPy Kotak 0.001", np_kecil)
ukur("Latihan 5: STRtree Kotak 0.001", lambda: [len(pohon.query(k, predicate="contains")) for k in kotak_kecil])
print()

# Latihan 6: Restore Delta Table
print("--- Latihan 6 ---")
dt_latihan = DeltaTable(PATH_DELTA, storage_options=opsi_drive)
print("versi sebelum restore:", dt_latihan.version(), "| baris:", len(dt_latihan.to_pandas()))

# Mengembalikan tabel ke keadaan versi 1
dt_latihan.restore(1)
dt_latihan = DeltaTable(PATH_DELTA, storage_options=opsi_drive)

print("versi setelah restore:", dt_latihan.version(), "| baris:", len(dt_latihan.to_pandas()))
print([h.get("operation") for h in dt_latihan.history()][:3])

--- Latihan 1 ---
[Latihan 1: SQL dengan indeks (2000 ID)] 0.0369 s
[Latihan 1: Key-value (2000 ID)] 0.0130 s
Rata-rata per pencarian -> SQL Indeks: 18.45 mikrodetik | Key-Value: 6.50 mikrodetik

--- Latihan 2 ---
Jumlah di TinyDB (Document): 146
Jumlah di SQLite (Tabel): 2180
Apakah sama? False

--- Latihan 3 ---
[Latihan 3: SUM 1 kolom (total_bayar)] 0.0241 s
[Latihan 3: SUM 2 kolom (jarak, total)] 0.0294 s

--- Latihan 4 ---
Graph (out-degree): [(30, 263), (46, 263), (52, 263), (55, 263), (74, 263)]
SQL (COUNT DISTINCT): [(30, 263), (46, 263), (52, 263), (55, 263), (74, 263)]
Apakah urutannya sama? True

--- Latihan 5 ---
[Latihan 5: NumPy Kotak 0.02] 0.5258 s
[Latihan 5: STRtree Kotak 0.02] 1.1109 s
[Latihan 5: NumPy Kotak 0.001] 0.4927 s
[Latihan 5: STRtree Kotak 0.001] 0.0136 s

--- Latihan 6 ---
versi sebelum restore: 24 | baris: 150000
versi setelah restore: 25 | baris: 150000
['RESTORE', 'RESTORE', 'OPTIMIZE']


In [12]:
# tugas praktikum

from deltalake import DeltaTable, write_deltalake
import pandas as pd
import os, shutil

# Persiapan Folder
PATH_TUGAS = f"{DIR_SIMPAN}/tugas_delta_2hari"
if os.path.exists(PATH_TUGAS):
    shutil.rmtree(PATH_TUGAS)

# HARI 1: Menulis data awal
print("--- HARI 1 ---")
data_hari_1 = df[KOLOM_DELTA].iloc[:10_000].copy()
write_deltalake(PATH_TUGAS, data_hari_1, mode="overwrite", storage_options=opsi_drive)
dt_tugas = DeltaTable(PATH_TUGAS, storage_options=opsi_drive)
print("Selesai Hari 1 -> Versi:", dt_tugas.version(), "| Total Baris:", len(dt_tugas.to_pandas()))

# HARI 2: Menambah data baru & Mengoreksi kesalahan
print("\n--- HARI 2 ---")
# 1. Menambah (append) 5.000 baris transaksi baru
data_hari_2 = df[KOLOM_DELTA].iloc[10_000:15_000].copy()
write_deltalake(PATH_TUGAS, data_hari_2, mode="append", storage_options=opsi_drive)

# 2. Koreksi data: Mengubah metode_bayar 'Tunai' menjadi 'Cash' khusus di wilayah Queens
dt_tugas = DeltaTable(PATH_TUGAS, storage_options=opsi_drive)
dt_tugas.update(
    predicate="wilayah_jemput = 'Queens' AND metode_bayar = 'Tunai'",
    updates={"metode_bayar": "'Cash'"}
)
print("Selesai Hari 2 -> Versi aktif:", DeltaTable(PATH_TUGAS, storage_options=opsi_drive).version())

# PEMBUKTIAN TIME TRAVEL & LOG TRANSAKSI
print("\n--- PEMBUKTIAN AUDIT ---")
# Membaca tabel persis seperti keadaan di akhir Hari 1 (Versi 0)
tabel_kemarin = DeltaTable(PATH_TUGAS, version=0, storage_options=opsi_drive).to_pandas()
print("Jumlah baris jika dibaca mundur ke Hari 1 (Versi 0):", len(tabel_kemarin))

# Memeriksa isi direktori log transaksi (Memastikan semua commit tercatat)
isi_log = sorted(os.listdir(f"{PATH_TUGAS}/_delta_log"))
print("\nIsi direktori _delta_log/:")
for berkas in isi_log:
    print("-", berkas)

--- HARI 1 ---
Selesai Hari 1 -> Versi: 0 | Total Baris: 10000

--- HARI 2 ---
Selesai Hari 2 -> Versi aktif: 1

--- PEMBUKTIAN AUDIT ---
Jumlah baris jika dibaca mundur ke Hari 1 (Versi 0): 10000

Isi direktori _delta_log/:
- 00000000000000000000.json
- 00000000000000000001.json
